# 11.7 · 命名实体识别 / Named Entity Recognition (NER)

> **课程定位 / Where this fits**
> 第 7 课，**Part 11 · 经典 NLP**。从"分类整段文本"转向"标注每个词"。
> Lesson 7, **Part 11 · Classic NLP**. From "classify whole text" to "label each token."
>
> **命名实体识别(NER)** 找出文本里的**实体**——人名(PER)、地名(LOC)、机构名(ORG)等，并定位它们的**边界**。它是信息抽取的基石：简历解析、知识图谱、问答、搜索都靠它。NER 是典型的**序列标注**任务——给句子里**每个词**打一个标签。本课讲清 **BIO 标注体系**、**特征工程**，**从零构造数据并训练一个特征式 NER 模型**，并用**实体级 F1**(而非词级准确率)正确评估。
> **Named Entity Recognition (NER)** finds **entities** in text — persons (PER), locations (LOC), organizations (ORG) — and their **boundaries**. The backbone of information extraction: resume parsing, knowledge graphs, QA, search. NER is a classic **sequence-labeling** task — tag **every token**. We cover the **BIO scheme**, **feature engineering**, **build data and train a feature-based NER model from scratch**, and evaluate correctly with **entity-level F1** (not token accuracy).
>
> 💼 **实战/面试视角**："BIO 标注 / NER 用什么特征 / 为什么用实体级F1 / CRF 作用" 是信息抽取岗常考。
> 💼 **Practical/interview angle:** "BIO tagging / NER features / why entity-level F1 / role of CRF" — info-extraction questions.

> 📐 **符号约定 / Notation**
> - 标签 B-/I-/O —— 实体开始/内部/非实体 / Begin/Inside/Outside an entity
> - 实体级 —— 以"整个实体跨度"为单位评估 / evaluate by whole entity spans

> 💡 **面试相关 / Interview-relevant**
> - "BIO/BIOES 标注体系"（出镜率 ★★★★★）
> - "NER 常用哪些特征"（★★★★，大小写/词缀/上下文/词典）
> - "为什么用实体级而非词级评估"（★★★★）
> - "CRF/BiLSTM-CRF 在 NER 的作用"（★★★★，标签转移约束）

---

## 学习目标 / Learning Objectives
1. 理解 NER 任务与 **BIO 标注体系**。
   Understand NER and the **BIO tagging scheme**.
2. 把 NER 建模成**序列标注**，设计**token 特征**。
   Frame NER as sequence labeling; design token features.
3. 训练一个特征式 NER 模型并预测。
   Train a feature-based NER model and predict.
4. 用**实体级 F1** 正确评估。
   Evaluate with entity-level F1.

## 目录 / TOC
1. [NER 与 BIO 标注 ⭐](#1)
2. [特征工程：怎么描述一个 token ⭐](#2)
3. [训练特征式 NER ⭐](#3)
4. [实体级评估 + 小结 ⭐](#4)


<a id="1"></a>
## 1. NER 与 BIO 标注 ⭐ / NER & BIO Tagging

实体常常**跨多个词**："New York"是一个地名(2 个词)，"Bank of China"是一个机构(3 个词)。怎么标注这种"跨度"？——**BIO 体系**：
Entities often **span multiple tokens**: "New York" is one location (2 words), "Bank of China" one org (3 words). How to tag spans? — the **BIO scheme**:
- **B-XXX**：实体的**第一个**词(Begin)。
  **B-XXX:** the **first** token of an entity (Begin).
- **I-XXX**：实体的**后续**词(Inside)。
  **I-XXX:** a **subsequent** token of the entity (Inside).
- **O**：不属于任何实体(Outside)。
  **O:** not part of any entity (Outside).

例如 "John Smith visited New York" → `B-PER I-PER O B-LOC I-LOC`。这样既标了**类别**又标了**边界**(B 表示新实体开始，区分相邻的两个同类实体)。
E.g. "John Smith visited New York" → `B-PER I-PER O B-LOC I-LOC`. This encodes both **type** and **boundary** (B marks a new entity, separating adjacent same-type entities).

> 还有 **BIOES**(加 E=结束, S=单词实体)等变体，更精细。NER 的本质：把"找实体"变成"给每个词分一个 BIO 标签"的**序列标注**问题。
> Variants like **BIOES** (E=End, S=Single) are finer. The essence: turn "find entities" into a **sequence-labeling** problem of assigning each token a BIO tag.

我们用**模板 + 实体词表**生成带 BIO 标注的句子(自包含)。
We generate BIO-tagged sentences from **templates + entity gazetteers** (self-contained).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
np.random.seed(0)

# 实体词表(gazetteers): 训练/测试用【不相交】的名字, 逼模型靠特征泛化而非死记 / disjoint train/test gazetteers
PER_TR = ["John Smith","Mary Jones","Alan Turing","Lisa Wang","Carlos Diaz","Anna Schmidt","David Kim","Sara Lee"]
LOC_TR = ["New York","Tokyo","Paris","London","San Francisco","Berlin","Cairo","Mumbai"]
ORG_TR = ["Google","Microsoft","NASA","Bank of China","United Nations","Stanford University","Toyota","Airbus"]
PER_TE = ["Robert Brown","Emma Watson","Hiro Tanaka","Olga Petrova","James Park","Nina Lopez"]   # 全新人名 / unseen
LOC_TE = ["Boston","Shanghai","Madrid","Sydney","Cape Town","Rio de Janeiro"]                     # 含"of/de"连接词 / lowercase connectors
ORG_TE = ["Amazon","Bank of America","University of Tokyo","World Health Organization","Sony","Intel"]
GAZ_TR = {"PER":PER_TR, "LOC":LOC_TR, "ORG":ORG_TR}
GAZ_TE = {"PER":PER_TE, "LOC":LOC_TE, "ORG":ORG_TE}
PER, LOC, ORG = PER_TR, LOC_TR, ORG_TR                    # 演示用训练词表 / for demo cells
# 句子模板(用占位符) / sentence templates with slots
TEMPLATES = [
    "{PER} works at {ORG} in {LOC} .",
    "{PER} visited {LOC} last summer .",
    "{ORG} announced a partnership with {ORG} yesterday .",
    "Yesterday {PER} flew from {LOC} to {LOC} .",
    "The CEO of {ORG} met {PER} in {LOC} .",
    "{PER} and {PER} joined {ORG} .",
]
def make_sentence(rng, gaz=GAZ_TR):
    tpl = rng.choice(TEMPLATES)
    tokens, tags = [], []
    for part in tpl.split():
        if part.startswith("{"):                          # 占位符 → 填一个实体并打 BIO 标签 / fill a slot
            etype = part.strip("{}")
            ent = rng.choice(gaz[etype]).split()
            for j, w in enumerate(ent):
                tokens.append(w); tags.append(("B-" if j==0 else "I-") + etype)  # 首词B 其余I / B then I
        else:
            tokens.append(part); tags.append("O")         # 普通词 → O / non-entity → O
    return tokens, tags

rng = np.random.RandomState(1)
toks, tags = make_sentence(rng)
print("BIO 标注示例:")
for w, t in zip(toks, tags): print(f"  {w:<14} {t}")
print("\nB-=实体首词, I-=实体内部词, O=非实体; New York→B-LOC I-LOC(一个跨度)")


<a id="2"></a>
## 2. 特征工程：怎么描述一个 token ⭐ / Feature Engineering

经典(非深度) NER 的关键是**特征**：用一组人工特征描述每个 token，让分类器据此预测它的 BIO 标签。哪些特征对识别实体有用？(面试常问)
The key to classic (non-deep) NER is **features**: describe each token with hand-crafted features so a classifier can predict its BIO tag. Which features help? (interview)
- **大小写**：实体(尤其人名/地名/机构)通常**首字母大写**——最强信号之一。
  **Capitalization:** entities are usually **capitalized** — one of the strongest signals.
- **词形**：全大写(NASA)、含数字、词缀(`-ton`,`-land` 常见于地名)。
  **Word shape:** all-caps (NASA), digits, suffixes (`-ton`, `-land` common in places).
- **上下文**：前一个词/后一个词(如 "in ___" 后常是地点，"at ___" 后常是机构)。
  **Context:** previous/next word ("in ___" → location, "at ___" → org).
- **词典(gazetteer)**：该词是否在已知实体表里。
  **Gazetteer:** is the word in a known-entity list.

> ⚠️ 单看一个词不够，**上下文特征**很关键——这也是为什么 NER 是**序列**任务(后面 11.8 的 HMM/CRF/BiLSTM 进一步建模标签之间的依赖)。
> ⚠️ One token alone isn't enough; **context features** matter — why NER is a **sequence** task (11.8's HMM/CRF/BiLSTM further model dependencies between labels).


In [ ]:
def word_features(tokens, i):
    """为句子中第 i 个 token 抽取特征 / features for token i in a sentence."""
    w = tokens[i]
    f = {
        "w.lower": w.lower(),
        "is_title": w.istitle(),                          # 首字母大写? / capitalized?
        "is_upper": w.isupper(),                          # 全大写? / all-caps?
        "suffix3": w[-3:].lower(),                        # 后三字符(词缀线索) / suffix
        "prefix2": w[:2].lower(),                         # 前两字符 / prefix
        "prev": tokens[i-1].lower() if i>0 else "<S>",    # 前一个词 / previous word
        "next": tokens[i+1].lower() if i<len(tokens)-1 else "</S>",  # 后一个词 / next word
        "prev_title": tokens[i-1].istitle() if i>0 else False,       # 前词是否大写(连续大写=多词实体) / prev cap
    }
    return f

print("token 'York'(在 'New York' 中) 的特征:")
for k, v in word_features(["John","visited","New","York","."], 3).items():
    print(f"  {k:<12} = {v}")
print("\n关键特征: is_title(大写) + prev_title(前词也大写→可能是多词实体) + 上下文(prev='new')")


<a id="3"></a>
## 3. 训练特征式 NER ⭐ / Training Feature-Based NER

把每个 token 的特征字典向量化(`DictVectorizer` 做 one-hot)，再用**逻辑回归**预测 BIO 标签。这就是 CRF 出现之前(以及很多轻量场景至今)的经典 NER 做法。
Vectorize each token's feature dict (`DictVectorizer` one-hot), then predict BIO tags with **logistic regression**. This is classic NER before CRF (and still used in lightweight settings).

为了检验**泛化**(而非死记词表)，测试句子里会出现**训练时没见过的新实体名**——模型得靠大小写/上下文/词缀等特征判断。
To test **generalization** (not memorization), test sentences include **entity names unseen in training** — the model must rely on capitalization/context/suffix features.


In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def gen_dataset(n, rng, gaz):
    sents = [make_sentence(rng, gaz) for _ in range(n)]
    X = [word_features(toks, i) for toks, _ in sents for i in range(len(toks))]  # 展平所有token特征 / flatten
    y = [t for _, tags in sents for t in tags]
    return sents, X, y

# 训练用训练词表; 测试用【完全不相交】的新实体名 → 检验泛化 / test uses disjoint unseen entities
_, Xtr_d, ytr = gen_dataset(600, np.random.RandomState(0), GAZ_TR)
test_sents, Xte_d, yte = gen_dataset(200, np.random.RandomState(123), GAZ_TE)
dv = DictVectorizer(sparse=True)
Xtr = dv.fit_transform(Xtr_d); Xte = dv.transform(Xte_d)        # 特征字典 → 稀疏 one-hot 向量 / vectorize
clf = LogisticRegression(max_iter=1000, C=5).fit(Xtr, ytr)
tok_acc = clf.score(Xte, yte)
print(f"特征式 NER: token 级准确率 = {tok_acc:.3f}  (特征向量维度 {Xtr.shape[1]})")

# 在一个新句子上预测 / predict on a fresh sentence
demo_tokens = "Lisa Wang flew from Berlin to Tokyo .".split()
demo_feats = dv.transform([word_features(demo_tokens, i) for i in range(len(demo_tokens))])
demo_pred = clf.predict(demo_feats)
print("\n预测演示:")
for w, t in zip(demo_tokens, demo_pred): print(f"  {w:<8} → {t}")


<a id="4"></a>
## 4. 实体级评估 + 小结 ⭐ / Entity-Level Evaluation

**为什么不能只看 token 准确率(面试重点)**：句子里大多数词是 `O`，全预测 `O` 也能有很高的 token 准确率，却一个实体都没抽对！正确做法是**实体级评估**：一个实体**只有边界和类型都完全正确**才算抽对，再算 **precision/recall/F1**。
**Why not token accuracy (interview):** most tokens are `O`, so predicting all `O` gives high token accuracy yet zero entities found! The right way is **entity-level evaluation**: an entity counts as correct **only if its boundary and type both exactly match**, then compute **precision/recall/F1**.


In [ ]:
def extract_entities(tokens, tags):
    """从 BIO 标签里抽出实体跨度 (类型, 起, 止) / extract entity spans from BIO tags."""
    ents = []; cur = None
    for i, t in enumerate(tags):
        if t.startswith("B-"):                            # 新实体开始 / new entity begins
            if cur: ents.append(cur)
            cur = [t[2:], i, i]
        elif t.startswith("I-") and cur and cur[0]==t[2:]:# 实体延续 / entity continues
            cur[2] = i
        else:                                             # O 或不连贯 → 结束当前实体 / O → close current
            if cur: ents.append(cur); cur = None
    if cur: ents.append(cur)
    return set(tuple(e) for e in ents)

# 在测试集上算实体级 P/R/F1 / entity-level P/R/F1 on test set
tp = fp = fn = 0; offset = 0
for toks, gold in test_sents:
    n = len(toks)
    pred = list(clf.predict(dv.transform([word_features(toks, i) for i in range(n)])))
    gold_e = extract_entities(toks, gold); pred_e = extract_entities(toks, pred)
    tp += len(gold_e & pred_e)                            # 边界+类型都对 / exact matches
    fp += len(pred_e - gold_e); fn += len(gold_e - pred_e)
prec = tp/(tp+fp); rec = tp/(tp+fn); f1 = 2*prec*rec/(prec+rec)
print(f"实体级评估: precision={prec:.3f}, recall={rec:.3f}, F1={f1:.3f}")
print(f"(对比 token 级准确率 {tok_acc:.3f} —— token级会被大量 O 抬高, 实体级才反映真实抽取能力)")

# 可视化: 把预测实体在句子里高亮 / highlight predicted entities
fig, ax = plt.subplots(figsize=(11, 1.8)); ax.axis("off")
colors = {"PER":"#2a9d8f","LOC":"#e9c46a","ORG":"#e76f51"}
toks2 = "David Kim and Sara Lee joined NASA .".split()
pred2 = clf.predict(dv.transform([word_features(toks2,i) for i in range(len(toks2))]))
x = 0.02
for w, t in zip(toks2, pred2):
    c = colors.get(t[2:], None) if t!="O" else None
    ax.text(x, 0.5, w, fontsize=13, bbox=dict(facecolor=c, alpha=0.6, boxstyle="round") if c else None)
    x += 0.018*len(w) + 0.04
ax.set_title("NER 预测高亮: 绿=人名 黄=地点 红=机构"); plt.tight_layout(); plt.show()
print("特征式NER靠 大小写+上下文+词缀 泛化到没见过的名字; 现代用 BiLSTM-CRF / BERT")


```
NER: 找文本中的实体(PER/LOC/ORG)及边界; 是序列标注(给每个token打标签)
BIO标注: B-首词/I-内部词/O-非实体; 既标类型又标边界(B区分相邻同类实体); 变体BIOES
特征(经典NER): 大小写(最强)/全大写/词缀/上下文前后词/词典gazetteer
建模: token特征→DictVectorizer→分类器(LogReg); 上下文特征关键(故NER是序列任务)
评估: 必须实体级P/R/F1(边界+类型全对才算); token准确率被大量O抬高=误导
现代: BiLSTM-CRF(CRF建模标签转移约束, 如I-PER不能跟在B-LOC后) / BERT微调最佳
```

### 💡 面试速查 / Interview cheat-sheet
1. **BIO 标注**: B-/I-/O 标类型+边界; NER=序列标注。
   BIO: B-/I-/O encode type+boundary; NER = sequence labeling.
2. **关键特征**: 大小写/词缀/上下文词/词典(经典NER靠特征工程)。
   Key features: capitalization/affixes/context/gazetteer.
3. **实体级评估**: 边界+类型全对才算; 别用token准确率(被O抬高)。
   Entity-level eval: exact span+type match; not token accuracy (inflated by O).
4. **CRF 作用**: 建模标签转移约束(I-PER 不接 B-LOC), 输出更连贯。
   CRF: models label-transition constraints for coherent output.
5. **现代 NER**: BiLSTM-CRF / BERT 微调。
   Modern NER: BiLSTM-CRF / fine-tuned BERT.

### 下一节 / Next
**11.8 序列标注**——NER 是序列标注的一种。本课系统讲序列标注(以词性标注为例)，并**从零实现 HMM + Viterbi 算法**——理解"如何利用标签之间的依赖"做最优序列解码，这是 CRF/结构化预测的基础。
**11.8 Sequence Labeling** — NER is one kind. This lesson covers sequence labeling systematically (via PoS tagging) and **implements HMM + the Viterbi algorithm from scratch** — understanding how to exploit label dependencies for optimal decoding, the basis of CRF/structured prediction.
